## Capital Edge Data Pipeline

In [1]:
# import libraries
import requests
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
from dotenv import load_dotenv  
import os

## EXTRACTION PROCESS


In [2]:
# in order not to hardcode the API key, we will load it from an environment variable

load_dotenv()  
API_KEY = os.getenv('API_KEY')

In [3]:
API_KEY

'QCQ9QHPVM69738OO'

In [11]:
# Defining endpoint URL

symbol = "IBM"
url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={API_KEY}"


In [12]:
# make a request to the server using API

response = requests.get(url)
response.status_code

200

In [13]:
# getting data in json format

data = response.json()
data



{'Meta Data': {'1. Information': 'Daily Prices (open, high, low, close) and Volumes',
  '2. Symbol': 'IBM',
  '3. Last Refreshed': '2026-06-12',
  '4. Output Size': 'Compact',
  '5. Time Zone': 'US/Eastern'},
 'Time Series (Daily)': {'2026-06-12': {'1. open': '278.9400',
   '2. high': '278.9400',
   '3. low': '267.6800',
   '4. close': '272.2400',
   '5. volume': '5748915'},
  '2026-06-11': {'1. open': '268.0000',
   '2. high': '276.4799',
   '3. low': '266.5000',
   '4. close': '274.8500',
   '5. volume': '7558322'},
  '2026-06-10': {'1. open': '273.8200',
   '2. high': '280.5100',
   '3. low': '271.3300',
   '4. close': '272.3600',
   '5. volume': '5497137'},
  '2026-06-09': {'1. open': '281.1350',
   '2. high': '283.5900',
   '3. low': '271.2900',
   '4. close': '277.4900',
   '5. volume': '9007918'},
  '2026-06-08': {'1. open': '286.4400',
   '2. high': '290.5000',
   '3. low': '279.4300',
   '4. close': '280.8200',
   '5. volume': '6790639'},
  '2026-06-05': {'1. open': '300.0000'

In [14]:
### checking to extract only the time series data excluding the metadata

data['Time Series (Daily)']

{'2026-06-12': {'1. open': '278.9400',
  '2. high': '278.9400',
  '3. low': '267.6800',
  '4. close': '272.2400',
  '5. volume': '5748915'},
 '2026-06-11': {'1. open': '268.0000',
  '2. high': '276.4799',
  '3. low': '266.5000',
  '4. close': '274.8500',
  '5. volume': '7558322'},
 '2026-06-10': {'1. open': '273.8200',
  '2. high': '280.5100',
  '3. low': '271.3300',
  '4. close': '272.3600',
  '5. volume': '5497137'},
 '2026-06-09': {'1. open': '281.1350',
  '2. high': '283.5900',
  '3. low': '271.2900',
  '4. close': '277.4900',
  '5. volume': '9007918'},
 '2026-06-08': {'1. open': '286.4400',
  '2. high': '290.5000',
  '3. low': '279.4300',
  '4. close': '280.8200',
  '5. volume': '6790639'},
 '2026-06-05': {'1. open': '300.0000',
  '2. high': '302.3000',
  '3. low': '281.0701',
  '4. close': '284.8400',
  '5. volume': '12509480'},
 '2026-06-04': {'1. open': '307.4250',
  '2. high': '310.4400',
  '3. low': '300.1800',
  '4. close': '301.7700',
  '5. volume': '9608097'},
 '2026-06-03

In [18]:
#extracting the first timeseries data to check the structure

first_date = next(iter(data['Time Series (Daily)']))
data['Time Series (Daily)'][first_date]

{'1. open': '278.9400',
 '2. high': '278.9400',
 '3. low': '267.6800',
 '4. close': '272.2400',
 '5. volume': '5748915'}

In [ ]:
# extracting the values for the low price

low_price = data['Time Series (Daily)'][first_date]['3. low']
low_price

'267.6800'

In [20]:
# extracting the values for the high price

high_price = data['Time Series (Daily)'][first_date]['2. high']
high_price

'278.9400'

###### The above data structure cannot be used for analysis, thus, its need to be transformed to a structured format in rows and columns, to fit into postgres and also aids analysis. Therefore, we need to go through the transformation process with the use of Pandas.

## TRANSFORMATION PROCESS


In [22]:
# creating a transform function that extracts the time series data and returns a dataframe and its becomes a modular reusable blocks of codes
#  for any stock symbol in any pipeline

def transform_time_series_data(data):

    # extracting the timeseries from jason data
    time_series = data['Time Series (Daily)']

    # converting the dictionary timeseries data into a dataframe
    df = pd.DataFrame.from_dict(time_series, orient='index')

    # Resetting the index to make the date a column
    df.reset_index(inplace=True)

    # rename index column to "date"
    df.rename(columns={'index': 'date'}, inplace=True)

    #convert the date column to datetime format 
    df['date'] = pd.to_datetime(df['date'])

    # Renaming  the columns for simplicity, removing the numbers and dots
    df.rename(columns={
        '1. open': 'open',
        '2. high': 'high',
        '3. low': 'low',
        '4. close': 'close',
        '5. volume': 'volume'
    }, inplace=True)


    # converting the numericcolumns to coreect data types
    df = df.astype({
        'open': 'float',
        'high': 'float',
        'low': 'float',
        'close': 'float',
        'volume': 'int'
    })

    # sort by date (important for time series data)
    df = df.sort_values('date').reset_index(drop=True)

    # remove duplicates if any
    df = df.drop_duplicates()

    return df

    

In [23]:
# call the transform function to get the dataframe

df = transform_time_series_data(data)
df.head(20)

,date,open,high,low,close,volume
0,2026-01-21,292.76,297.670,292.5100,297.54,5185023
1,2026-01-22,299.42,300.930,293.5300,294.67,3670152
2,2026-01-23,294.07,294.335,289.7900,292.44,3298424
3,2026-01-26,293.16,296.815,293.1400,296.33,3726890
4,2026-01-27,297.16,297.330,293.2700,293.86,2944722
5,2026-01-28,294.17,295.950,291.2601,294.16,5790347
6,2026-01-29,317.86,319.900,303.4700,309.24,10124929
7,2026-01-30,307.60,307.783,299.7300,306.70,5940669
8,2026-02-02,307.51,316.640,306.4100,314.73,4581189
9,2026-02-03,312.40,312.975,283.8500,294.31,11296020


### DATA LOADING

In [24]:
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

In [25]:
DB_NAME

'capitaledge_db'

In [26]:
# CREATING A CONNECTION URL

db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# creating engine using sqlalchemy
engine = create_engine(db_url)

#load dataframe to sql database
df.to_sql('stock_prices', engine, if_exists='append', index=False)

print("Data loaded successfully to the database!")


Data loaded successfully to the database!


In [27]:
# save dataframe as csv file
df.to_csv('stock_prices.csv', index=False)

print("Data saved successfully to stock_prices.csv!")

Data saved successfully to stock_prices.csv!
